<a href="https://colab.research.google.com/github/Moquiuti/LangGraph_Orquestrando_agentes_e_multiagentes/blob/main/integrar_mem%C3%B3ria_Langgraph.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q -U langgraph
!pip install -q -U langchain
!pip install -q -U langchain-core
!pip install -q -U langchain-google-genai
!pip install -q -U pydantic
!pip install -q -U python-dotenv

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 234.3/234.3 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 548.1/548.1 kB 16.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.0/56.0 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.0/41.0 kB 1.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langchain 1.2.15 requires langgraph<1.2.0,>=1.1.5, but you have langgraph 1.2.0 which is incompatible.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.3/114.3 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.6/67.6 kB 1.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 109.4/109.4 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 472.3/472.3 kB 11.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 23.3 MB/s eta 0:00:00
ERROR: p

In [2]:
import os
import uuid
import json
from typing import TypedDict, Optional, Dict, Any, List

from dotenv import load_dotenv
from pydantic import BaseModel, Field

from langchain_core.messages import HumanMessage, SystemMessage, AIMessage
from langchain_core.tools import tool
from langchain_google_genai import ChatGoogleGenerativeAI

from langgraph.graph import StateGraph, END
from langgraph.store.memory import InMemoryStore
from langgraph.prebuilt import create_react_agent

In [3]:
try:
    from google.colab import userdata
    os.environ["GEMINI_API_KEY"] = userdata.get("GEMINI_API_KEY")
    print("GEMINI_API_KEY carregada pelo Colab Secrets.")
except Exception:
    load_dotenv()
    print("Tentando carregar variáveis pelo .env.")

if not os.getenv("GEMINI_API_KEY"):
    raise ValueError("GEMINI_API_KEY não encontrada.")

GEMINI_API_KEY carregada pelo Colab Secrets.


In [4]:
llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0
)

In [5]:
memory_store = InMemoryStore()

In [6]:
USER_ID = "leandro"
MEMORY_NAMESPACE = ("email_assistant", USER_ID)

In [7]:
perfil_usuario = {
    "nome": "Leandro Moquiuti",
    "cargo": "Desenvolvedor Backend Java Sênior",
    "empresa": "Marlabs",
    "tom_resposta": "profissional, objetivo, educado e direto",
    "prioridades": [
        "assuntos técnicos de projetos",
        "reuniões com cliente",
        "demandas de liderança",
        "arquitetura de software",
        "atividades urgentes do time"
    ]
}

regras_triagem = {
    "responder": [
        "quando o e-mail exigir retorno direto",
        "quando houver pergunta técnica",
        "quando for solicitação de cliente, liderança ou time",
        "quando pedir posicionamento, validação ou confirmação"
    ],
    "notificar": [
        "quando o e-mail for importante, mas não exigir resposta imediata",
        "quando for apenas informativo, porém relevante",
        "quando mencionar prazo, reunião, cliente ou bloqueio"
    ],
    "ignorar": [
        "quando for propaganda",
        "quando for newsletter genérica",
        "quando não tiver relação com trabalho ou prioridade",
        "quando não exigir nenhuma ação"
    ]
}

In [8]:
memory_store.put(
    MEMORY_NAMESPACE,
    "preferencia_tom",
    {
        "tipo": "preferencia",
        "conteudo": "O usuário prefere respostas profissionais, objetivas, educadas e diretas."
    }
)

memory_store.put(
    MEMORY_NAMESPACE,
    "preferencia_reuniao",
    {
        "tipo": "preferencia",
        "conteudo": "O usuário prefere reuniões entre 09:00 e 17:00."
    }
)

memory_store.put(
    MEMORY_NAMESPACE,
    "contexto_profissional",
    {
        "tipo": "perfil",
        "conteudo": "O usuário atua como desenvolvedor backend Java sênior e costuma lidar com integrações técnicas, clientes, liderança e arquitetura de software."
    }
)

In [9]:
@tool
def SearchMemoryTool(query: str) -> str:
    """
    Busca memórias relevantes sobre o usuário para contextualizar respostas de e-mail.
    Use esta ferramenta antes de responder quando precisar lembrar preferências, perfil ou histórico.
    """
    resultados = memory_store.search(
        MEMORY_NAMESPACE,
        query=query,
        limit=5
    )

    if not resultados:
        return "Nenhuma memória relevante encontrada."

    memorias = []

    for item in resultados:
        memorias.append({
            "key": item.key,
            "value": item.value
        })

    return json.dumps(memorias, ensure_ascii=False, indent=2)

In [10]:
@tool
def ManagedMemoryTool(key: str, content: str, memory_type: str = "observacao") -> str:
    """
    Cria ou atualiza uma memória do usuário.
    Use quando descobrir uma preferência, padrão de resposta ou informação recorrente útil para e-mails futuros.
    """
    memory_store.put(
        MEMORY_NAMESPACE,
        key,
        {
            "tipo": memory_type,
            "conteudo": content
        }
    )

    return f"Memória salva com sucesso na chave: {key}"

In [11]:
acoes_executadas = []

In [12]:
@tool
def WriteMail(destinatario: str, assunto: str, corpo: str) -> str:
    """
    Cria um rascunho de e-mail. Não envia e-mail real.
    """
    acao = {
        "tipo": "rascunho_email",
        "destinatario": destinatario,
        "assunto": assunto,
        "corpo": corpo
    }

    acoes_executadas.append(acao)

    return f"Rascunho criado para {destinatario} com assunto '{assunto}'."

In [13]:
@tool
def CheckCalendar(data: str) -> str:
    """
    Consulta disponibilidade simulada no calendário.
    """
    return "Disponibilidade simulada: hoje às 16:00 ou amanhã às 10:00."

In [14]:
@tool
def ScheduleMeeting(participante: str, data_hora: str, assunto: str) -> str:
    """
    Agenda uma reunião simulada. Não cria evento real.
    """
    acao = {
        "tipo": "reuniao",
        "participante": participante,
        "data_hora": data_hora,
        "assunto": assunto
    }

    acoes_executadas.append(acao)

    return f"Reunião simulada criada com {participante} em {data_hora}."

In [15]:
tools = [
    SearchMemoryTool,
    ManagedMemoryTool,
    WriteMail,
    CheckCalendar,
    ScheduleMeeting
]

In [17]:
#Construção dinâmica do prompt
#Vou destacar aqui sendo a parte mais importante deste exercício...

#vou montar um prompt e tentar combinar nessa situação
#perfil do usuário, regras de triagem, memórias salvas, histórico resumido, e-mail atual

def buscar_memorias_contextuais(query: str) -> str:
    resultados = memory_store.search(
        MEMORY_NAMESPACE,
        query=query,
        limit=5
    )

    if not resultados:
        return "Nenhuma memória prévia encontrada."

    linhas = []

    for item in resultados:
        valor = item.value
        linhas.append(f"- {item.key}: {valor.get('conteudo', valor)}")

    return "\n".join(linhas)

In [20]:
def montar_prompt_dinamico(email: Dict[str, Any], historico: List[str]) -> str:
    query_memoria = f"""
    Assunto: {email.get("assunto")}
    Corpo: {email.get("corpo")}
    Remetente: {email.get("remetente")}
    """

    memorias = buscar_memorias_contextuais(query_memoria)

    historico_texto = "\n".join(historico[-5:]) if historico else "Sem histórico anterior nesta conversa."

    prompt = f"""
Você é um assistente inteligente de e-mails para o usuário abaixo.

PERFIL DO USUÁRIO:
{json.dumps(perfil_usuario, ensure_ascii=False, indent=2)}

REGRAS DE TRIAGEM:
{json.dumps(regras_triagem, ensure_ascii=False, indent=2)}

MEMÓRIAS RELEVANTES DO USUÁRIO:
{memorias}

HISTÓRICO RECENTE DA THREAD:
{historico_texto}

E-MAIL ATUAL:
Remetente: {email.get("remetente")}
Destinatário: {email.get("destinatario")}
Assunto: {email.get("assunto")}
Data: {email.get("data_recebimento")}

Corpo:
{email.get("corpo")}

INSTRUÇÕES:
- Use as memórias relevantes para personalizar a resposta.
- Use SearchMemoryTool se precisar consultar mais contexto.
- Use ManagedMemoryTool se descobrir uma nova preferência útil para e-mails futuros.
- Use WriteMail apenas para criar rascunho, nunca para enviar.
- Se o e-mail sugerir reunião, use CheckCalendar antes de ScheduleMeeting.
- Responda de forma profissional, objetiva e contextualizada.
- Não invente informações.
"""
    return prompt

In [22]:
email_memory_agent = create_react_agent(
    model=llm,
    tools=tools
)

/tmp/ipykernel_4676/3712709211.py:1: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  email_memory_agent = create_react_agent(


In [24]:
class EmailMemoryState(TypedDict):
    email: Dict[str, Any]
    historico: List[str]
    prompt_dinamico: Optional[str]
    resposta: Optional[str]

In [26]:
def no_montar_prompt(state: EmailMemoryState):
    prompt = montar_prompt_dinamico(
        email=state["email"],
        historico=state.get("historico", [])
    )

    return {
        "prompt_dinamico": prompt
    }

In [28]:
def no_executar_agente(state: EmailMemoryState):
    resultado = email_memory_agent.invoke({
        "messages": [
            HumanMessage(content=state["prompt_dinamico"])
        ]
    })

    resposta = resultado["messages"][-1].content

    historico = list(state.get("historico", []))
    historico.append(f"E-mail: {state['email'].get('assunto')}")
    historico.append(f"Resposta: {resposta}")

    return {
        "resposta": resposta,
        "historico": historico
    }

In [30]:
builder = StateGraph(EmailMemoryState)

builder.add_node("montar_prompt", no_montar_prompt)
builder.add_node("executar_agente", no_executar_agente)

builder.set_entry_point("montar_prompt")

builder.add_edge("montar_prompt", "executar_agente")
builder.add_edge("executar_agente", END)

grafo_email_memoria = builder.compile()

In [31]:
email_evento = {
    "remetente": "ana.cliente@empresa.com",
    "destinatario": "leandro@empresa.com",
    "assunto": "Dúvida sobre integração B2B com fornecedor",
    "corpo": """
Olá, Leandro.

Conseguimos avançar na análise da integração B2B, mas surgiu uma dúvida sobre o fluxo de autenticação
e sobre qual endpoint será usado para consultar disponibilidade.

Você consegue nos responder ainda hoje ou sugerir um horário para alinharmos rapidamente?

Obrigado.
""",
    "data_recebimento": "2026-05-18 14:30"
}

In [34]:
#testar fluxo completo

estado_inicial = {
    "email": email_evento,
    "historico": [],
    "prompt_dinamico": None,
    "resposta": None
}

resultado = grafo_email_memoria.invoke(estado_inicial)

print(resultado["resposta"])

[{'type': 'text', 'text': 'Rascunho de e-mail criado e salvo.', 'extras': {'signature': 'CroEAQw51scRUKfgBORySo3UmGOvE2QOGS825sX2vVqcT0dXsotysNLPZIB1VBfuJ7cU3KjoG+b6FukzYat8OsF3Y0SaCcCl76xg1HhC1M+t+oa0nMq0U6cLGY7s1U8X96U/UaT9gC6YzSC/S0EH/mksTUZmgxNClrpngiAMsG7QFr1KL2eelJjK7i47yR51GgiEZMIN3M1eF4j6bQbMIEHuEPXYnQTqQpGAviH3E9xMep/JP0hwOhuLJsDw4MKgsM8qQX09dC2Wg/7qSG3ioW45ed+kj+xgk+jxzcbs5rU4hyxYRgF1RQWoQrKGC5OGi6Rsh93D/et695CKcAJvO2Fu50AL6OGzFUDqxOf3RjhbVjVxX+dOCsVQOanoWzc32G8W4p1dA/2yHARMp3k6uNK0EQFgzBKGE5jqmnAPhnJCAK+MdQ38NTwnlCXlP2fZwKsIhfzhyBj9EHezgp+PLramkGZg3K3eRnOrJ30P6hKxILn6xUuBwP+g4GQCwbdwoiuKIt5lSr5DLe5UskMDy+rdqrwcI89N5939Dp6rgkr+KjNRotZkZH+5Cd//ydUnmZyBTFlrNq6pLkwjlIZNXx+uk2s/e9s3uVjEX0dD5Y+SrD9Y9EDqu/NRuAKTQEjIboYklEpK8Pxo9JRJbSKOJZ/axUfeFSqyaUL7nWHKYpSjzj7XAVSvmYO4M49be23aPmUVObpRZB93Jef67sLtwU64byvUlGLbrNhRPwlyaBhSSBgQkciXUfjWMDoG'}}]


In [35]:
#ver ações executadas
json.dumps(acoes_executadas, ensure_ascii=False, indent=2)

'[\n  {\n    "tipo": "rascunho_email",\n    "destinatario": "ana.cliente@empresa.com",\n    "assunto": "Re: Dúvida sobre integração B2B com fornecedor",\n    "corpo": "Olá, Ana.\\n\\nRecebi sua mensagem e entendo a dúvida sobre o fluxo de autenticação e o endpoint para consulta de disponibilidade na integração B2B.\\n\\nTenho disponibilidade para um alinhamento rápido hoje às 16:00. Por favor, me informe se este horário funciona para você. Caso contrário, posso adiantar algumas informações por e-mail.\\n\\nÀ disposição.\\n\\nAtenciosamente,\\nLeandro Moquiuti"\n  }\n]'

In [36]:
email_evento_2 = {
    "remetente": "ana.cliente@empresa.com",
    "destinatario": "leandro@empresa.com",
    "assunto": "Re: Dúvida sobre integração B2B com fornecedor",
    "corpo": """
Leandro, obrigada pelo retorno.

Pode considerar então que amanhã às 10h seria um bom horário para fazermos esse alinhamento?
Também gostaria que você priorizasse uma explicação mais objetiva para o time de negócio.

Obrigada.
""",
    "data_recebimento": "2026-05-18 15:10"
}

In [37]:
resultado_2 = grafo_email_memoria.invoke({
    "email": email_evento_2,
    "historico": resultado["historico"],
    "prompt_dinamico": None,
    "resposta": None
})

print(resultado_2["resposta"])

[{'type': 'text', 'text': 'Rascunho de e-mail criado e salvo.', 'extras': {'signature': 'CtoCAQw51sfqX4KQFA7dWFQGDojTMP+HFd+9wWs5vhrSC8pvWsntaBCgEl9gB5R6VdELnp/KXLo5eBokJIotYquIztDysLBkxQ8HXMjpT7PtWijoKEQtll9HcNVIRLE9R3vbYvaBS4N29DkZmKxeGkX11q0h8N3FPAz6Ht20elICOrjZ9dRAf72a6nd/JqJg29dMqT6X3aOpvPPjf5AdwY+W3xBjL93cQ1oYQIkMvveVBV0o3KK43G6aiEY/0x1FbMc5o24DZ1F+SpzuPRpJA4feVxtDb4lPe3l1w5V1BpJxW7OJZS6Ay9UoP60WmYF3BZaYY5T3sPY+2Lwr62po8O5+JufduZCNio73BkC26DuCccJwrCJ4V+BbKjjdne262nTF80D7gDD1GarFHi8qGJ80QWsXFxvbiGI8BB0GFKBPj+dxfrwOm9bufUvrVjiPAwoCePiMD407lnw9Pg=='}}]


In [38]:
#salvar nova preferência manualmente

ManagedMemoryTool.invoke({
    "key": "preferencia_cliente_ana",
    "content": "A cliente Ana prefere explicações objetivas para repassar ao time de negócio.",
    "memory_type": "preferencia_cliente"
})

'Memória salva com sucesso na chave: preferencia_cliente_ana'

In [39]:
#agora testar outro email da mesma cliente

email_evento_3 = {
    "remetente": "ana.cliente@empresa.com",
    "destinatario": "leandro@empresa.com",
    "assunto": "Resumo para time de negócio",
    "corpo": """
Leandro, você consegue me mandar um resumo simples da integração para eu apresentar ao time de negócio?
""",
    "data_recebimento": "2026-05-18 16:00"
}

In [40]:
resultado_3 = grafo_email_memoria.invoke({
    "email": email_evento_3,
    "historico": resultado_2["historico"],
    "prompt_dinamico": None,
    "resposta": None
})

print(resultado_3["resposta"])

Rascunho de e-mail criado e salvo.


In [41]:
memorias = memory_store.search(
    MEMORY_NAMESPACE,
    query="cliente Ana explicação objetiva negócio",
    limit=10
)

for memoria in memorias:
    print(memoria.key, "->", memoria.value)

preferencia_tom -> {'tipo': 'preferencia', 'conteudo': 'O usuário prefere respostas profissionais, objetivas, educadas e diretas.'}
preferencia_reuniao -> {'tipo': 'preferencia', 'conteudo': 'O usuário prefere reuniões entre 09:00 e 17:00.'}
contexto_profissional -> {'tipo': 'perfil', 'conteudo': 'O usuário atua como desenvolvedor backend Java sênior e costuma lidar com integrações técnicas, clientes, liderança e arquitetura de software.'}
preferencia_cliente_ana -> {'tipo': 'preferencia_cliente', 'conteudo': 'A cliente Ana prefere explicações objetivas para repassar ao time de negócio.'}


In [42]:
#versão um pouco mais economica

prompt_teste = montar_prompt_dinamico(email_evento, [])

print(prompt_teste[:2000])


Você é um assistente inteligente de e-mails para o usuário abaixo.

PERFIL DO USUÁRIO:
{
  "nome": "Leandro Moquiuti",
  "cargo": "Desenvolvedor Backend Java Sênior",
  "empresa": "Marlabs",
  "tom_resposta": "profissional, objetivo, educado e direto",
  "prioridades": [
    "assuntos técnicos de projetos",
    "reuniões com cliente",
    "demandas de liderança",
    "arquitetura de software",
    "atividades urgentes do time"
  ]
}

REGRAS DE TRIAGEM:
{
  "responder": [
    "quando o e-mail exigir retorno direto",
    "quando houver pergunta técnica",
    "quando for solicitação de cliente, liderança ou time",
    "quando pedir posicionamento, validação ou confirmação"
  ],
  "notificar": [
    "quando o e-mail for importante, mas não exigir resposta imediata",
    "quando for apenas informativo, porém relevante",
    "quando mencionar prazo, reunião, cliente ou bloqueio"
  ],
  "ignorar": [
    "quando for propaganda",
    "quando for newsletter genérica",
    "quando não tiver rela